In [57]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

In [58]:
LAT = 31.2001
LON = 29.9187

url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    "?parameters=T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
    "&community=RE"
    f"&latitude={LAT}"
    f"&longitude={LON}"
    "&start=20210101"
    "&end=20251231"
    "&format=JSON"
)

## API Documentation

**Endpoint:** `https://power.larc.nasa.gov/api/temporal/daily/point` (NASA POWER Daily Point API)

**Region:** Alexandria, Egypt (`lat=31.2001`, `lon=29.9187`)

**Parameters requested:**
- `T2M` — mean temperature at 2m (°C)
- `T2M_MAX` / `T2M_MIN` — max/min temperature at 2m (°C)
- `PRECTOTCORR` — bias-corrected precipitation (mm/day)
- `RH2M` — relative humidity at 2m (%)
- `WS2M` — wind speed at 2m (m/s)
- `ALLSKY_SFC_SW_DWN` — all-sky surface shortwave downward irradiance (kWh/m²/day)
- `community=RE` — renewable energy community parameter set

**Date range:** `2021-01-01` to `2025-12-31` (5 full years, daily granularity). A multi-year daily range was chosen (rather than a short hourly window) so that trend, yearly seasonality, and decomposition could be analyzed properly.

**Rate limits:** none hit. NASA POWER's point API requires no API key and returned a `200` response on the first request; no throttling or retry logic was needed for this single call.


In [59]:
response = requests.get(url)
print(response.status_code)

200


In [60]:
data = response.json()
print(data.keys())

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [61]:
parameters = data["properties"]["parameter"]

temperature = parameters["T2M"]
temp_max = parameters["T2M_MAX"]
temp_min = parameters["T2M_MIN"]
precipitation = parameters["PRECTOTCORR"]
humidity = parameters["RH2M"]
wind_speed = parameters["WS2M"]
solar_radiation = parameters["ALLSKY_SFC_SW_DWN"]

temperature

{'20210101': 16.61,
 '20210102': 17.1,
 '20210103': 17.25,
 '20210104': 17.25,
 '20210105': 17.84,
 '20210106': 17.36,
 '20210107': 17.9,
 '20210108': 17.56,
 '20210109': 17.27,
 '20210110': 18.05,
 '20210111': 18.17,
 '20210112': 17.63,
 '20210113': 16.75,
 '20210114': 15.33,
 '20210115': 14.15,
 '20210116': 14.4,
 '20210117': 14.19,
 '20210118': 13.89,
 '20210119': 12.8,
 '20210120': 13.0,
 '20210121': 12.66,
 '20210122': 13.4,
 '20210123': 13.9,
 '20210124': 14.4,
 '20210125': 14.7,
 '20210126': 14.48,
 '20210127': 15.26,
 '20210128': 13.35,
 '20210129': 13.42,
 '20210130': 13.87,
 '20210131': 15.8,
 '20210201': 17.17,
 '20210202': 17.02,
 '20210203': 16.75,
 '20210204': 17.11,
 '20210205': 16.36,
 '20210206': 16.39,
 '20210207': 16.37,
 '20210208': 15.92,
 '20210209': 16.82,
 '20210210': 16.04,
 '20210211': 15.87,
 '20210212': 15.39,
 '20210213': 15.31,
 '20210214': 14.63,
 '20210215': 15.6,
 '20210216': 11.7,
 '20210217': 10.66,
 '20210218': 12.55,
 '20210219': 12.82,
 '20210220':

In [62]:
df = pd.DataFrame({
    "date": temperature.keys(),
    "temperature": temperature.values(),
    "max_temperature": temp_max.values(),
    "min_temperature": temp_min.values(),
    "precipitation": precipitation.values(),
    "humidity": humidity.values(),
    "wind_speed": wind_speed.values(),
    "solar_radiation": solar_radiation.values()
})

In [63]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [64]:
df.set_index("date", inplace=True)

In [65]:
df.head()

,temperature,max_temperature,min_temperature,precipitation,humidity,wind_speed,solar_radiation
date,,,,,,,
2021-01-01,16.61,21.28,13.44,0.0,71.51,2.41,3.5100
2021-01-02,17.10,21.77,13.56,0.0,76.06,2.57,3.2546
2021-01-03,17.25,22.67,13.85,0.0,81.02,2.51,3.1934
2021-01-04,17.25,22.45,13.60,0.0,80.47,1.76,3.1932
2021-01-05,17.84,23.25,14.76,0.0,83.66,1.76,3.0451


In [66]:
df.to_csv("alexandria_weather_2021-2025.csv")

In [67]:
import json

with open("alexandria_weather_raw.json", "w") as f:
    json.dump(data, f, indent=4)